In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import warnings

from pathlib import Path
import sys
import os
from datetime import datetime
from dataScraper import *

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

# from src.utils.team_info import nameDict
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])
df = df.sort_values(["PLAYER_ID", "GAME_DATE"]).reset_index(drop=True)
 
# Derived features
df["PTS_PER_MIN"] = df["PTS"] / df["MIN"].replace(0, np.nan)
df["IS_HOME"]     = df["MATCHUP"].str.contains("vs\\.").astype(int)
df["SPREAD_PROXY"] = df["TEAM_PLUS_MINUS"]  # actual outcome; use betting spread at prediction time

starting = pd.read_csv('src/points_model/sp_checkpoint.csv')
starting = starting[['PLAYER_ID', 'START_POSITION']]
df = starting.merge(df, on='PLAYER_ID', how='left')
df.head()

,PLAYER_ID,START_POSITION,SEASON_YEAR,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,...,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,PTS_PER_MIN,IS_HOME,SPREAD_PROXY
0,1642262,F,2025-26,Cody Williams,Cody,1.610613e+09,UTA,Utah Jazz,22500087.0,2025-10-22,...,0.523,0.555,96.5,95.0,79.17,95.0,0.379,0.000000,1.0,21.0
1,1642262,F,2025-26,Cody Williams,Cody,1.610613e+09,UTA,Utah Jazz,22500135.0,2025-10-29,...,0.589,0.630,111.7,110.5,92.08,110.0,0.489,0.335383,1.0,-2.0
2,1642262,F,2025-26,Cody Williams,Cody,1.610613e+09,UTA,Utah Jazz,22500025.0,2025-10-31,...,0.566,0.596,103.0,101.5,84.58,102.0,0.633,0.538237,0.0,-22.0
3,1642262,F,2025-26,Cody Williams,Cody,1.610613e+09,UTA,Utah Jazz,22500150.0,2025-11-02,...,0.591,0.645,102.7,101.0,84.17,101.0,0.606,0.000000,0.0,-23.0
4,1642262,F,2025-26,Cody Williams,Cody,1.610613e+09,UTA,Utah Jazz,22500036.0,2025-11-07,...,0.665,0.686,107.4,106.5,88.75,107.0,0.689,0.517241,0.0,-40.0


In [3]:
def rolling_player(df, col, windows=[3, 5, 10], min_periods=1):
    """Per-player rolling means (shifted to avoid leakage)."""
    out = {}
    for w in windows:
        out[f"{col}_roll{w}"] = (
            df.groupby("PLAYER_ID")[col]
            .transform(lambda x: x.shift(1).rolling(w, min_periods=min_periods).mean())
        )
    return out
 
for col in ["MIN", "PTS_PER_MIN", "USG_PCT", "PACE"]:
    for k, v in rolling_player(df, col).items():
        df[k] = v
 
# Exponential weighted mean (λ decay, equivalent to span≈7 games)
df["MIN_ewm"]         = df.groupby("PLAYER_ID")["MIN"].transform(lambda x: x.shift(1).ewm(span=7).mean())
df["PTS_PER_MIN_ewm"] = df.groupby("PLAYER_ID")["PTS_PER_MIN"].transform(lambda x: x.shift(1).ewm(span=7).mean())
 
# Games played (sample size tracker)
df["GP"] = df.groupby("PLAYER_ID").cumcount()  # 0-indexed; GP=0 means first game
 
# Blowout risk proxy: rolling team point differential variance
df["BLOWOUT_RISK"] = df.groupby("PLAYER_ID")["TEAM_PLUS_MINUS"].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).std()
)

df['STARTER_FLAG'] = df['START_POSITION'].notna().astype(int)

In [4]:
# Drop stale prior columns so re-running this cell does not merge into prior_mean_x / prior_mean_y
_prior_stale = [
    c
    for c in df.columns
    if c in ("prior_mean", "prior_std")
    or c.startswith("prior_mean_")
    or c.startswith("prior_std_")
]
if _prior_stale:
    df = df.drop(columns=_prior_stale)

league_prior = (
    df.groupby("STARTER_FLAG")["PTS_PER_MIN"]
    .agg(prior_mean="mean", prior_std="std")
    .reset_index()
)
df = df.merge(league_prior, on="STARTER_FLAG", how="left")

# Conjugate normal–normal posterior mean (vectorized)
prior_m = df["prior_mean"]
prior_v = (df["prior_std"] ** 2).clip(lower=1e-12)

obs_mean = df["PTS_PER_MIN_ewm"].where(df["PTS_PER_MIN_ewm"].notna(), prior_m)
obs_n = df["GP"].clip(upper=20)
obs_var = prior_v

prior_prec = 1 / prior_v
obs_prec = obs_n / obs_var
posterior_mean = (prior_prec * prior_m + obs_prec * obs_mean) / (prior_prec + obs_prec)

df["BAYES_PTS_PER_MIN"] = np.where(obs_n.to_numpy() == 0, prior_m.to_numpy(), posterior_mean.to_numpy())
df.head()

,PLAYER_ID,START_POSITION,SEASON_YEAR,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,...,PACE_roll5,PACE_roll10,MIN_ewm,PTS_PER_MIN_ewm,GP,BLOWOUT_RISK,STARTER_FLAG,prior_mean,prior_std,BAYES_PTS_PER_MIN
0,1642262,F,2025-26,Cody Williams,Cody,1.610613e+09,UTA,Utah Jazz,22500087.0,2025-10-22,...,NaN,NaN,NaN,NaN,0,NaN,1,0.503847,0.257487,0.503847
1,1642262,F,2025-26,Cody Williams,Cody,1.610613e+09,UTA,Utah Jazz,22500135.0,2025-10-29,...,100.000000,100.000000,2.400000,0.000000,1,NaN,1,0.503847,0.257487,0.251923
2,1642262,F,2025-26,Cody Williams,Cody,1.610613e+09,UTA,Utah Jazz,22500025.0,2025-10-31,...,102.320000,102.320000,7.843810,0.191647,2,16.263456,1,0.503847,0.257487,0.295714
3,1642262,F,2025-26,Cody Williams,Cody,1.610613e+09,UTA,Utah Jazz,22500150.0,2025-11-02,...,104.276667,104.276667,10.879279,0.341524,3,21.517435,1,0.503847,0.257487,0.382105
4,1642262,F,2025-26,Cody Williams,Cody,1.610613e+09,UTA,Utah Jazz,22500036.0,2025-11-07,...,102.337500,102.337500,10.265143,0.216624,4,20.728402,1,0.503847,0.257487,0.274068


In [5]:
MIN_FEATURES = [
    "MIN_roll3", "MIN_roll5", "MIN_roll10",
    "MIN_ewm",
    "STARTER_FLAG",
    "BLOWOUT_RISK",
    "IS_HOME",
    "PACE_roll3",
    "GP",
]
 
# Drop rows with NaN in features or target
min_df = df[MIN_FEATURES + ["MIN", "GAME_DATE"]].dropna()
 
# ── TIME-BASED SPLIT ──
split_date = min_df["GAME_DATE"].quantile(0.75)  # train on first 75% of season
train_mask = min_df["GAME_DATE"] <= split_date
val_mask   = ~train_mask
 
X_train_min = min_df[MIN_FEATURES][train_mask]
y_train_min = min_df["MIN"][train_mask]
X_val_min   = min_df[MIN_FEATURES][val_mask]
y_val_min   = min_df["MIN"][val_mask]
 
scaler_min = StandardScaler()
X_train_min_s = scaler_min.fit_transform(X_train_min)
X_val_min_s   = scaler_min.transform(X_val_min)
 
ridge_min = Ridge(alpha=10.0)
ridge_min.fit(X_train_min_s, y_train_min)
 
min_preds_val = ridge_min.predict(X_val_min_s)
min_mae = mean_absolute_error(y_val_min, min_preds_val)
print(f"[Minutes Model]  Validation MAE: {min_mae:.2f} min")

[Minutes Model]  Validation MAE: 4.78 min


In [6]:
PPM_FEATURES = [
    "BAYES_PTS_PER_MIN",
    "PTS_PER_MIN_roll3", "PTS_PER_MIN_roll5",
    "PTS_PER_MIN_ewm",
    "USG_PCT_roll3",
    "PACE_roll3",
    "IS_HOME",
    "STARTER_FLAG",
    "GP",
    # Opponent defense
    "OPP_DEF_RATING",
]
 
# Check column exists
PPM_FEATURES = [f for f in PPM_FEATURES if f in df.columns]
 
ppm_df = df[PPM_FEATURES + ["PTS_PER_MIN", "GAME_DATE"]].replace([np.inf, -np.inf], np.nan).dropna()
 
split_date_ppm = ppm_df["GAME_DATE"].quantile(0.75)
train_mask_ppm = ppm_df["GAME_DATE"] <= split_date_ppm
val_mask_ppm   = ~train_mask_ppm
 
X_train_ppm = ppm_df[PPM_FEATURES][train_mask_ppm]
y_train_ppm = ppm_df["PTS_PER_MIN"][train_mask_ppm]
X_val_ppm   = ppm_df[PPM_FEATURES][val_mask_ppm]
y_val_ppm   = ppm_df["PTS_PER_MIN"][val_mask_ppm]
 
scaler_ppm = StandardScaler()
X_train_ppm_s = scaler_ppm.fit_transform(X_train_ppm)
X_val_ppm_s   = scaler_ppm.transform(X_val_ppm)
 
ridge_ppm = Ridge(alpha=5.0)
ridge_ppm.fit(X_train_ppm_s, y_train_ppm)
 
ppm_preds_val = ridge_ppm.predict(X_val_ppm_s)
ppm_mae = mean_absolute_error(y_val_ppm, ppm_preds_val)
print(f"[Pts/Min Model]  Validation MAE: {ppm_mae:.4f} pts/min")

[Pts/Min Model]  Validation MAE: 0.1875 pts/min


In [7]:
# Reconstruct val set projection
val_common = ppm_df[val_mask_ppm].copy()
val_common["PRED_PTS_PER_MIN"] = ppm_preds_val
 
val_min_aligned = min_df[val_mask].copy()
val_min_aligned["PRED_MIN"] = min_preds_val
 
# Merge on index (both share original df index ordering)
# Simpler: recompute on shared rows
combined_idx = val_common.index.intersection(val_min_aligned.index)
val_combined = val_common.loc[combined_idx].copy()
val_combined["PRED_MIN"]         = val_min_aligned.loc[combined_idx, "PRED_MIN"]
val_combined["TRUE_PTS"]         = df.loc[combined_idx, "PTS"]
val_combined["PRED_PTS"]         = val_combined["PRED_MIN"] * val_combined["PRED_PTS_PER_MIN"]
 
pts_mae  = mean_absolute_error(val_combined["TRUE_PTS"], val_combined["PRED_PTS"])
pts_bias = (val_combined["PRED_PTS"] - val_combined["TRUE_PTS"]).mean()
print(f"[Points Proj]    Validation MAE:  {pts_mae:.2f} pts")
print(f"[Points Proj]    Bias (pred-true): {pts_bias:.2f} pts")
 

[Points Proj]    Validation MAE:  4.56 pts
[Points Proj]    Bias (pred-true): -0.42 pts


In [11]:
def project_player(player_name: str, opp_def_rating: float = None, spread: float = None):
    """
    Project points for a player's next game.
    opp_def_rating: opponent defensive rating (lower = tougher defense)
    spread: positive = player's team favored
    """
    player_rows = df[df["PLAYER_NAME"].str.lower() == player_name.lower()].copy()
    if player_rows.empty:
        # Try partial match
        mask = df["PLAYER_NAME"].str.lower().str.contains(player_name.lower())
        player_rows = df[mask].copy()
    if player_rows.empty:
        return f"Player '{player_name}' not found."
 
    latest = player_rows.sort_values("GAME_DATE").iloc[-1]
 
    # ── Minutes projection ──
    min_row = {f: latest.get(f, np.nan) for f in MIN_FEATURES}
    min_row_df = pd.DataFrame([min_row])
    min_row_scaled = scaler_min.transform(min_row_df.fillna(min_row_df.mean()))
    proj_min = float(ridge_min.predict(min_row_scaled)[0])
 
    # Blowout adjustment on minutes
    if spread is not None and abs(spread) > 15:
        proj_min *= 0.90  # -10% for likely blowout
 
    # ── Pts/Min projection ──
    ppm_row = {f: latest.get(f, np.nan) for f in PPM_FEATURES}
    if opp_def_rating is not None:
        ppm_row["OPP_DEF_RATING"] = opp_def_rating
    ppm_row_df = pd.DataFrame([ppm_row])
    ppm_row_scaled = scaler_ppm.transform(ppm_row_df.fillna(ppm_row_df.mean()))
    proj_ppm = float(ridge_ppm.predict(ppm_row_scaled)[0])
 
    proj_pts = proj_min * proj_ppm
 
    # ── Output ──
    print(f"\n{'─'*45}")
    print(f"  Player:          {latest['PLAYER_NAME']}")
    print(f"  Last game:       {latest['GAME_DATE'].date()}  {latest['MATCHUP']}  {latest['PTS']} pts in {latest['MIN']:.1f} min")
    print(f"  Recent avg min:  {latest.get('MIN_roll5', np.nan):.1f}  (last 5 games)")
    print(f"  Starter flag:    {'Yes' if latest.get('STARTER_FLAG') else 'No'}")
    print(f"  Bayesian ppm:    {latest.get('BAYES_PTS_PER_MIN', np.nan):.3f}")
    print(f"  ── Projections ──")
    print(f"  Projected MIN:   {proj_min:.1f}")
    print(f"  Projected ppm:   {proj_ppm:.3f}")
    print(f"  Projected PTS:   {proj_pts:.1f}")
    print(f"{'─'*45}\n")
    return proj_pts

print("\n=== SAMPLE PROJECTIONS (next game) ===")
for name in ["Tyler Herro"]:
    project_player(name)


=== SAMPLE PROJECTIONS (next game) ===

─────────────────────────────────────────────
  Player:          Tyler Herro
  Last game:       2026-03-19  MIA vs. LAL  21.0 pts in 32.7 min
  Recent avg min:  32.1  (last 5 games)
  Starter flag:    Yes
  Bayesian ppm:    0.652
  ── Projections ──
  Projected MIN:   30.8
  Projected ppm:   0.642
  Projected PTS:   19.8
─────────────────────────────────────────────

